# Ajuste de hiperparámetros y comparación con CNN simple

**Objetivo:** ajustar los hiperparámetros de **SVM** y **Random Forest** mediante validación cruzada agrupada anidada (`GridSearchCV` + `StratifiedGroupKFold`) y compararlos en igualdad de condiciones contra una **CNN simple entrenada desde cero** en PyTorch sobre los recortes RGB `224x224` de `preparar_cnn`.

Todas las comparaciones mantienen intactos los **mismos tres pliegues externos** del cuaderno 06 para que ningún fruto ni campaña de captura aparezca en entrenamiento y validación al mismo tiempo.

In [14]:
from pathlib import Path
import json
import sys

candidatos = [Path.cwd(), *Path.cwd().parents]
PROYECTO = next((p for p in candidatos if (p / 'data/processed/base_revisada_v2/base_svm_rf.csv').exists()), None)
if PROYECTO is None:
    PROYECTO = next((
        p / 'proyecto_estado_frutas' for p in candidatos
        if (p / 'proyecto_estado_frutas/data/processed/base_revisada_v2/base_svm_rf.csv').exists()
    ), None)
if PROYECTO is None:
    raise FileNotFoundError('Ejecutar primero la preparación de datos hasta generar base_revisada_v2/base_svm_rf.csv.')
if str(PROYECTO) not in sys.path:
    sys.path.insert(0, str(PROYECTO))

import pandas as pd
from IPython.display import Image, Markdown, display
from src.models.evaluacion import cargar_base_v2
from src.training.entrenar_modelos import ejecutar_comparacion

SALIDA = PROYECTO / 'experiments/results/comparacion_modelos_v2'
CHECKPOINTS = PROYECTO / 'experiments/checkpoints/comparacion_v2'
pd.set_option('display.max_columns', 15)
pd.set_option('display.precision', 4)

base, columnas, estado = cargar_base_v2(PROYECTO)
print(f'Base vigente: {len(base)} imágenes | {len(columnas)} características | {base.grupo_particion.nunique()} grupos.')

FileNotFoundError: Ejecutar primero la preparación de datos hasta generar base_revisada_v2/base_svm_rf.csv.

In [ ]:

import pandas as pd
from IPython.display import Image, Markdown, display
from src.models.evaluacion import cargar_base_v2
from src.training.entrenar_modelos import ejecutar_comparacion

SALIDA = PROYECTO / 'experiments/results/comparacion_modelos_v2'
CHECKPOINTS = PROYECTO / 'experiments/checkpoints/comparacion_v2'
pd.set_option('display.max_columns', 15)
pd.set_option('display.precision', 4)

base, columnas, estado = cargar_base_v2(PROYECTO)
print(f'Base vigente: {len(base)} imágenes | {len(columnas)} características | {base.grupo_particion.nunique()} grupos.')

ModuleNotFoundError: No module named 'src'

## 1. Protocolo de validación anidada sin fuga de grupos

- **Pliegues externos:** `StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=42)`.
- **SVM:** `StandardScaler` dentro del `Pipeline`, búsqueda en rejilla sobre `kernel` (`rbf`, `linear`), `C` y `gamma`, con `class_weight='balanced'` y `probability=True`.
- **Random Forest:** búsqueda sobre `n_estimators`, `max_depth`, `min_samples_leaf`, `max_features` y `class_weight`.
- **CNN simple (PyTorch):** 3 bloques convolucionales (`Conv2d + BatchNorm2d + ReLU + MaxPool2d`) + `AdaptiveAvgPool2d(4, 4)` + capas densas con `Dropout`, entrenada desde cero con pérdida `CrossEntropyLoss` ponderada por frecuencia de clase y aumento de datos ligero exclusivo en entrenamiento.

En la evaluación externa de la CNN, la configuración óptima y la duración de épocas se seleccionan únicamente dentro de la validación interna agrupada (`train_idx`), nunca usando `val_idx` para elegir pesos ni hiperparámetros.

In [ ]:
tablas = ejecutar_comparacion(PROYECTO, incluir_cnn=True, max_epocas_cnn=25)
display(tablas['metricas_globales'])
display(tablas['metricas_pliegues'])

NameError: name 'ejecutar_comparacion' is not defined

## 2. Hiperparámetros seleccionados y curvas de aprendizaje

Revisamos los mejores hiperparámetros encontrados en la búsqueda agrupada y las curvas de pérdida y F1 macro por época de la CNN para analizar el comportamiento de generalización y posible sobreajuste.

In [ ]:
resumen_ml = json.loads((SALIDA / 'resumen_ajuste_ml.json').read_text(encoding='utf-8'))
for modelo, info in resumen_ml.items():
    print(f'=== {modelo} ===')
    print('Mejores parámetros globales:', info['mejores_parametros_global'])
    print('F1 macro CV interno:', round(info['f1_macro_cv_interno_global'], 4))

cnn_global = pd.read_csv(SALIDA / 'cnn_busqueda_global.csv')
print('=== CNN simple (búsqueda global interna) ===')
display(cnn_global)
display(Image(filename=str(SALIDA / 'curvas_entrenamiento_cnn.png')))

## 3. Comparación global, matrices de confusión y desglose por clase y fruta

In [ ]:
display(Image(filename=str(SALIDA / 'comparacion_modelos.png')))
display(Image(filename=str(SALIDA / 'matrices_confusion.png')))
display(tablas['metricas_por_clase'])
display(tablas['metricas_por_fruta'])

## 4. Verificación de artefactos guardados

In [ ]:
import joblib
import torch

print('Checkpoints en:', CHECKPOINTS)
for p in sorted(CHECKPOINTS.iterdir()):
    print('-', p.name)

rf_pkg = joblib.load(CHECKPOINTS / 'random_forest_ajustado.joblib')
cnn_pkg = torch.load(CHECKPOINTS / 'cnn_calidad.pt', map_location='cpu', weights_only=True)
assert rf_pkg['sha_base'] == estado['sha_base']
assert cnn_pkg['sha_base'] == estado['sha_base']

mejor = tablas['metricas_globales'].query("modelo != 'Referencia_mayoritaria'").sort_values('f1_macro', ascending=False).iloc[0]
display(Markdown(
    f"**Resultado OOF principal:** `{mejor.modelo}` alcanza F1 macro de **{mejor.f1_macro:.3f}** "
    f"y recall macro de **{mejor.recall_macro:.3f}** sobre las 291 imágenes en 3 pliegues agrupados."
))